# P-2026-11-26-1: Reporter-level reassignment of HS 999999 to *Energy*

**Question** (from `proposals/2026-11-25.md` #1, in the open 11-25 PR): Saudi Arabia files its oil as HS 999999 ("commodities not specified") in 2019, 2020 and 2023, and as ch. 27 in 2021. Qatar does the same in 2020 and 2022. The lab mapping puts 999999 in *Services and Utilities* **and** in the *High-complexity Products* aggregate. If we move 999999 into *Energy Drilling and Mining* for those reporter-years only, in memory, (a) do their baskets and ranks start to look like their own ch. 27 years, and (b) does grouped ECI improve under the pre-stated rule A?

**Why tonight:** `QUEUE.md` Q01–Q16 are all taken on `main`. Every open `claude/nightly-*` branch up to 11-25 has taken the top idea from the previous night's proposals. Under PROTOCOL §1.4, tonight comes from `proposals/`. 11-25 #1 is untaken: no run folder on `main` or any `claude/nightly-*` branch names it. 11-25 found that no *chapter-mapping* change fixes Saudi Arabia, and this is the *reporter-level* fix it pointed to.

- **Date:** 2026-11-26 (America/New_York), nightly run. ID `P-2026-11-26-1`.
- **Data:** `data/raw/2019–2023_mega.csv` (HS6). Every variant is rebuilt from this with the lab chapter mapping (`lib/research.LAB_GROUPS`). `data/cleaned/2019–2023clean.csv` (cleaned grouped base, for the sample definition only); `external-data/raw/atlas_hs92_country_year.csv` (ATLAS ECI, benchmark only); `external-data/raw/wdi_panel.csv` (GDPpc PPP). No downloads.
- **Years:** 2019 primary; 2020–2022 as checks; 2023 shown but suspect (D1).
- **Library:** `lib/eci.py` and `lib/research.py`, imported unchanged. `o4`, the joint common sample and the F1 mapping are copied from 11-24/11-25, so tonight's numbers are directly comparable.
- **Run time:** printed in the last cell (about 15 s).

## TL;DR

- **The reassignment fixes the Saudi data problem exactly as intended.** Under the primary rule (RS), Saudi Arabia's 2019 basket becomes {Energy, Chemicals, Raw Manufacturing}, **identical** to its own 2021 (ch. 27-year) basket. Its grouped rank goes **4 → 48/105** (ATLAS 45). The 2020 move is 4 → 49 (ATLAS 46) and the 2023 move is 3 → 55/109 (ATLAS 49). The built-in check passes.
- **It still fails rule A, because income fit falls.** RS raises agreement with ATLAS in 2019 (ΔM1 **+0.025** [0.002, 0.051]). But it *lowers* the income fit (ΔM2 **−0.008** [−0.051, 0.032] in 2019, −0.046 in 2020 and in 2022). Both metrics improve in 0 of 3 check years. RM, RK and F1+RS fail as well. **F1 alone is still the only variant that passes.**
- **Why the income fit falls:** the reassignment moves rich oil exporters to where ATLAS puts them, and that is a long way below their income rank. In 2019 Saudi Arabia is 15th of 105 on log GDPpc, 4th on R0 and 48th on RS. In 2022 Qatar is 4th on income, 11th on R0 and 104th on RS (ATLAS 57). With Saudi Arabia excluded from the scoring, 2019 ΔM2 turns positive (+0.005). The "error" was helping M2. This is the resource-rich pattern from Maitree's findings acting on the metric.
- **Qatar is over-corrected.** Under RS, its 2022 basket is *Energy* only (rank 104/105, ATLAS 57). In the lab sample, Qatar appears only in 2022 (other years have a zero group), so there is no own clean year to check it against.
- **For the reports:** RS changes only 1 peer position across 2019–2022 for the 7 Spring 2026 countries: UAE 2nd → 1st among its peers in 2022 (Qatar falls). F1+RS changes 4. In the UAE peer group under R0, grouped ECI ranks Saudi Arabia above the UAE in 2019–2021, while ATLAS ranks the UAE first every year. RS does not change the 2019–2021 ordering (SAU > KWT > ARE), so the UAE report's peer sentence is not fixed by this.

## Why this matters for the lab

Every report prints a grouped-ECI rank and compares it with three peers. The UAE report's peers are Saudi Arabia, Kuwait and Qatar, and two of the three switch how they file their oil across years (11-25 C2b). If the cleaning step (`code/data-retrieval/datacleaning2025.ipynb`, owned by Will) were to reassign 999999 for such reporters at the next data update, would ECI get better or worse by the lab's own yardstick? Agenda §3 (assumptions of our implementation: the data say what the group names say) and §5 (score every variant under a rule fixed in advance).

## Method (pre-stated before any variant was scored)

**Base:** lab grouped rules (drop-any-zero filter, strict `RCA > 1`, sorted SVD, O4 orientation), on the **joint common sample** of 11-21/11-24/11-25 (cleaned grouped base ∩ HS4 base ∩ ATLAS ∩ WDI ∩ resource share: 105 countries in 2019). **R0** = lab mapping rebuilt from `data/raw`.

**Reassignment:** for a flagged reporter-year, the reporter's whole 999999 value is moved into ch. 27 (so into *Energy*). It leaves *Services and Utilities* and the *High-complexity Products* aggregate. Nothing else changes, and other reporters are untouched.

**Sample handling (pre-stated):** after reassignment, a flagged reporter's *Services* cell is 0 (999999 is the only ch. 99 line). The drop-any-zero filter would then delete it, which defeats the purpose. So the filter is applied to the **un-reassigned** matrix, and that fixes the row set. The reassigned matrix is then scored on those rows. Cell B prints which reporters a re-applied filter would drop.

**Flag rules:**
- **RS (primary, "convention switch")**: 999999 ≥ 50% of the reporter's exports in year *y*, **and** ch. 27 ≥ 50% of its exports in at least one *other* repo year. This targets reporters that demonstrably file the same oil under ch. 27 in other years.
- **RM (the proposal's literal rule)**: 999999 ≥ 50% and ch. 27 < 1% in year *y*.
- **RK**: Saudi Arabia only (999999 ≥ 50%).
- **F1+RS**: RS on top of 11-24's F1 mapping (ch. 89 → Transportation). **F1** is re-scored as a reference.

**Metrics and rule** (unchanged since 11-19): **M1** = Spearman with ATLAS ECI, **M2** = Spearman with log GDPpc (PPP). **Rule A:** a variant is *better* only if it beats R0 on both metrics in 2019 **and** in ≥2 of 2020–2022. The 2019 paired country bootstrap uses 2,000 draws.

**Pre-stated diagnostics:** (C) the built-in check, i.e. the flagged baskets vs the reporter's own ch. 27 year (Saudi Arabia 2021). (D3) scoring with the flagged reporters excluded, plus how far *other* countries move. (E) M cells that flip outside the flagged rows (the denominator effect from 11-25 C3). (F) Spring 2026 report countries and their peer positions.

*Added after seeing D/D3 (labelled as such):* G compares the flagged reporters' income rank with their R0 / RS / ATLAS / HS4 ranks.

## Setup

In [1]:
import sys, time, copy, warnings
T0 = time.time()
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
import eci, research as R
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30); pd.set_option('display.precision', 3)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
YEARS = [2019, 2020, 2021, 2022, 2023]; CHK = [2019, 2020, 2021, 2022]
AGG, SVC, ENE = 'High-complexity Products', 'Services and Utilities', 'Energy Drilling and Mining'
PEERS = {'CAN': ['USA', 'AUS', 'GBR'], 'DEU': ['FRA', 'GBR', 'ITA'], 'POL': ['DEU', 'CZE', 'HUN'],
         'SGP': ['KOR', 'JPN', 'DEU'], 'ARE': ['SAU', 'KWT', 'QAT'], 'GBR': ['DEU', 'FRA', 'ITA'], 'VNM': ['SGP', 'MYS', 'PHL']}

def o4(M):
    """Sorted SVD of binary M, O4 orientation (PCI negative with ubiquity). Copied unchanged from 11-24/11-25."""
    Mv = M.to_numpy(float); kc, kp = Mv.sum(1), Mv.sum(0)
    B = Mv / np.sqrt(kc[:, None] * kp[None, :])
    U, S, Vt = np.linalg.svd(B, full_matrices=False)
    k = U[:, 1] / np.sqrt(kc); q = Vt[1] / np.sqrt(kp)
    s4 = -np.sign(np.corrcoef(q, kp)[0, 1]); k, q = k * s4, q * s4
    e = pd.Series(k, index=M.index); z = (e - e.mean()) / e.std()
    return z, e, pd.Series(q, index=M.columns), S[1]

def lab_M(X, keep=None):
    """Lab rules: drop any country with a zero group, strict RCA > 1.
    If ``keep`` is given, the drop-any-zero filter is NOT re-applied: rows are restricted to ``keep``
    (the countries that pass the filter under the un-reassigned matrix). See Method."""
    X = eci.notebook_filter(X) if keep is None else X.loc[[i for i in keep if i in X.index]]
    Mm = (eci.rca(X) > 1).astype(float)
    return Mm.loc[Mm.sum(1) > 0, Mm.sum(0) > 0]

def hs_M(X):
    Mm = (eci.rca(X) >= 1).astype(float)
    return Mm.loc[Mm.sum(1) > 0, Mm.sum(0) > 0]

def rank(e, idx):
    return e[idx].rank(ascending=False)

G = {y: R.lab_groups_iso3(y, drop_eu=True) for y in YEARS}
XH = {y: R.repo_hs(y, 4) for y in YEARS}
RS = {y: XH[y][[c for c in XH[y].columns if c[:2] in ['26', '27', '71']]].sum(1) / XH[y].sum(1) for y in YEARS}
AT = R.atlas_cy(); ATL = {y: AT.query('year == @y').set_index('iso3').eci for y in YEARS}
LNY = {y: np.log(R.gdppc(y, 'ppp')).dropna() for y in YEARS}
GB = {y: o4(lab_M(G[y])) for y in YEARS}; HB = {y: o4(hs_M(XH[y])) for y in YEARS}
COMMON = {y: sorted(set(GB[y][0].index) & set(HB[y][0].index) & set(ATL[y].dropna().index) & set(LNY[y].index) & set(RS[y].index)) for y in YEARS}
print('joint common sample n (same definition as 11-21/11-24/11-25):', {y: len(COMMON[y]) for y in YEARS})

joint common sample n (same definition as 11-21/11-24/11-25): {2019: 105, 2020: 105, 2021: 94, 2022: 105, 2023: 109}


**Takeaway:** the sample matches 11-24/11-25 exactly (105 / 105 / 94 / 105 / 109).

## Results

### A. Which reporter-years does each rule flag?

In [2]:
# A. Reporter-year shares of ch.27 and of HS 999999 in total exports (raw HS6, M49 aggregates excluded)
SH = []
for y in YEARS:
    raw = R.raw_hs6(y); raw = raw[[c for c in raw.columns if c not in eci.M49_AGGREGATES]]
    tot = raw.sum(); ch = raw.index.str[:2]
    u = raw.loc['999999'] if '999999' in raw.index else pd.Series(0.0, index=raw.columns)
    d = pd.DataFrame({'year': y, 'm49': raw.columns, 'total $B': (tot / 1e9).values,
                      'ch27 share': (raw[ch == '27'].sum() / tot).values, '999999 share': (u / tot).values, '999999 $B': (u / 1e9).values})
    SH.append(d[tot.values > 0])
SH = pd.concat(SH, ignore_index=True); SH['iso3'] = [eci.m49_to_iso3(c) for c in SH.m49]
# max ch.27 share in the reporter's OTHER repo years
SH['max ch27 share, other years'] = [SH[(SH.m49 == m) & (SH.year != y)]['ch27 share'].max() for m, y in zip(SH.m49, SH.year)]
# Pre-stated flag rules
SH['RULE_SWITCH'] = (SH['999999 share'] >= 0.5) & (SH['max ch27 share, other years'] >= 0.5)   # primary: convention switch
SH['RULE_MECH'] = (SH['999999 share'] >= 0.5) & (SH['ch27 share'] < 0.01)                       # proposal's literal "ch.27 = 0" rule
SH['RULE_SAU'] = SH.iso3.eq('SAU') & (SH['999999 share'] >= 0.5)                                 # Saudi Arabia only
show = SH[SH.RULE_SWITCH | SH.RULE_MECH | SH.RULE_SAU | (SH['999999 share'] >= 0.2)].sort_values(['iso3', 'year'])
print(show[['iso3', 'year', 'total $B', 'ch27 share', '999999 share', '999999 $B', 'max ch27 share, other years', 'RULE_SWITCH', 'RULE_MECH', 'RULE_SAU']].round(3).to_string(index=False))
FLAG = {r: {y: list(SH[(SH.year == y) & SH[r]].m49) for y in YEARS} for r in ['RULE_SWITCH', 'RULE_MECH', 'RULE_SAU']}
for r in FLAG:
    print(r, {y: [eci.m49_to_iso3(m) for m in FLAG[r][y]] for y in YEARS})

iso3  year  total $B  ch27 share  999999 share  999999 $B  max ch27 share, other years  RULE_SWITCH  RULE_MECH  RULE_SAU
 ALB  2019     2.719       0.006         0.533      1.450                        0.064        False       True     False
 ALB  2020     2.416       0.009         0.438      1.058                        0.064        False      False     False
 ALB  2021     3.562       0.000         0.579      2.063                        0.064        False       True     False
 ALB  2022     4.306       0.004         0.922      3.972                        0.064        False       True     False
 ALB  2023    34.999       0.064         0.294     10.283                        0.009        False      False     False
 ARG  2021    77.934       0.026         0.331     25.794                        0.088        False      False     False
 BHR  2021    18.199       0.001         0.318      5.787                        0.422        False      False     False
 BHR  2022    22.252       0.001

**Takeaway (A):** the convention-switch rule flags exactly the reporter-years 11-25 listed by hand: **Saudi Arabia 2019/2020/2023** (999999 = 80.4% / 71.2% / 77.4% of exports, ch. 27 = 0.000, but 0.742 in its other years) and **Qatar 2020/2022** (999999 = 84.7% / 76.8%; ch. 27 is 0.865 in its other years). The literal "ch. 27 = 0" rule (RM) misses Qatar, because Qatar still files 14.1% / 22.1% under ch. 27 in those years. RM also catches Albania (2019/2021/2022) and Bermuda (2022), which never file meaningful ch. 27 in any year (max 0.064 and 0.000), so there is no evidence that their 999999 is oil. No 2021 reporter is flagged by RS, so 2021 is a no-change year for RS and RK.

### B. Build the variants

In [3]:
# B. Variants. Reassignment = move the reporter's whole 999999 line into ch.27 (-> Energy), out of Services AND the aggregate.
def groups_reassigned(year, mapping, m49s):
    raw = R.raw_hs6(year).copy()
    for m in m49s:
        if m in raw.columns and '999999' in raw.index:
            v = raw.at['999999', m]
            if '279999' not in raw.index:
                raw.loc['279999'] = 0.0
            raw.at['279999', m] += v; raw.at['999999', m] = 0.0
    ch = raw.index.str[:2].astype(int)
    X = pd.DataFrame({g: raw[np.isin(ch, chs)].sum() for g, chs in mapping.items()})
    X = X.drop(index=[c for c in X.index if c in eci.M49_AGGREGATES])
    X.index = [eci.m49_to_iso3(c) for c in X.index]
    return X.loc[pd.notna(X.index)].groupby(level=0).sum().pipe(lambda d: d.loc[d.sum(1) > 0])

L0 = R.LAB_GROUPS
F1 = copy.deepcopy(L0); F1['Daily Instruments'] = [c for c in L0['Daily Instruments'] if c != 89]; F1['Transportation'] = L0['Transportation'] + [89]
VARS = {'R0 lab mapping (rebuilt)': (L0, None), 'F1 ch.89 -> Transportation (ref.)': (F1, None),
        'RS reassign, switch rule (primary)': (L0, 'RULE_SWITCH'), 'RM reassign, mechanical rule': (L0, 'RULE_MECH'),
        'RK reassign, Saudi Arabia only': (L0, 'RULE_SAU'), 'F1+RS': (F1, 'RULE_SWITCH')}
VN = list(VARS); V0 = VN[0]
XV, VAR, PCI, MM, KEEP = {}, {}, {}, {}, {}
for y in YEARS:
    for v, (mp, rule) in VARS.items():
        XV[y, v] = groups_reassigned(y, mp, FLAG[rule][y] if rule else [])
    for v, (mp, rule) in VARS.items():
        base = VN[1] if mp is F1 else V0
        KEEP[y, v] = list(eci.notebook_filter(XV[y, base]).index)   # sample fixed by the un-reassigned matrix
        MM[y, v] = lab_M(XV[y, v], keep=KEEP[y, v])
        z, k, q, s_ = o4(MM[y, v]); VAR[y, v] = z; PCI[y, v] = q
# Would the drop-any-zero filter remove flagged reporters if re-applied after reassignment?
for y in YEARS:
    for v in ['RS reassign, switch rule (primary)', 'RM reassign, mechanical rule']:
        lost = sorted(set(KEEP[y, v]) - set(eci.notebook_filter(XV[y, v]).index))
        if lost: print(f'{y} {v.split()[0]}: re-applying drop-any-zero after reassignment would drop {lost}')
CS = {(y, v): sorted(set(COMMON[y]) & set(VAR[y, V0].index) & set(VAR[y, v].index)) for y in YEARS for v in VN}
print('pairwise sample n:'); print(pd.DataFrame({v.split()[0]: [len(CS[y, v]) for y in YEARS] for v in VN}, index=YEARS).T.to_string())
print('R0 (rebuilt) vs cleaned-CSV base, Spearman:', {y: round(stats.spearmanr(VAR[y, V0][CS[y, V0]], GB[y][0][CS[y, V0]])[0], 3) for y in YEARS})

2019 RS: re-applying drop-any-zero after reassignment would drop ['SAU']
2019 RM: re-applying drop-any-zero after reassignment would drop ['SAU']
2020 RS: re-applying drop-any-zero after reassignment would drop ['SAU']
2020 RM: re-applying drop-any-zero after reassignment would drop ['SAU']
2022 RS: re-applying drop-any-zero after reassignment would drop ['QAT']
2022 RM: re-applying drop-any-zero after reassignment would drop ['BMU']
2023 RS: re-applying drop-any-zero after reassignment would drop ['SAU']
2023 RM: re-applying drop-any-zero after reassignment would drop ['SAU']
pairwise sample n:
       2019  2020  2021  2022  2023
R0      105   105    94   105   109
F1      105   105    94   105   109
RS      105   105    94   105   109
RM      105   105    94   105   109
RK      105   105    94   105   109
F1+RS   105   105    94   105   109
R0 (rebuilt) vs cleaned-CSV base, Spearman: {2019: np.float64(0.992), 2020: np.float64(0.994), 2021: np.float64(0.987), 2022: np.float64(1.0), 20

**Takeaway (B):** if the drop-any-zero filter were re-applied after reassignment, it would delete the very reporters being fixed: Saudi Arabia in 2019/2020/2023 and Qatar in 2022. Any cleaning-step version of this fix must also change the filter, or give those reporters a non-zero *Services* cell. With the row set fixed by R0, every variant has the same pairwise n as R0. The rebuilt R0 agrees with the cleaned-CSV base at Spearman 0.987–1.000.

### C. Built-in check: do reassigned baskets look like the reporter's own ch. 27 year?

In [4]:
# C. Built-in check: do the reassigned baskets of flagged reporters look like their own un-flagged years?
rows = []
for iso in ['SAU', 'QAT']:
    for y in YEARS:
        for v in [V0, 'RS reassign, switch rule (primary)']:
            M = MM[y, v]
            if iso not in M.index: rows.append({'iso3': iso, 'year': y, 'variant': v.split()[0], 'basket (RCA>1)': '(not in lab sample)'}); continue
            c = CS[y, v]
            rows.append({'iso3': iso, 'year': y, 'variant': v.split()[0], 'flagged': any(eci.m49_to_iso3(m) == iso for m in FLAG['RULE_SWITCH'][y]),
                         'basket (RCA>1)': ', '.join(M.columns[M.loc[iso] == 1]),
                         'rank': f"{int(rank(VAR[y, v], c)[iso])}/{len(c)}" if iso in c else '-',
                         'ATLAS rank': int(rank(ATL[y], c)[iso]) if iso in c else '-'})
CHKT = pd.DataFrame(rows); print(CHKT.to_string(index=False))

iso3  year variant flagged                                                                 basket (RCA>1)    rank  ATLAS rank
 SAU  2019      R0    True                    Chemicals, High-complexity Products, Services and Utilities   4/105        45.0
 SAU  2019      RS    True                       Energy Drilling and Mining, Chemicals, Raw Manufacturing  48/105        45.0
 SAU  2020      R0    True Chemicals, Raw Manufacturing, High-complexity Products, Services and Utilities   4/105        46.0
 SAU  2020      RS    True                       Energy Drilling and Mining, Chemicals, Raw Manufacturing  49/105        46.0
 SAU  2021      R0   False                       Energy Drilling and Mining, Chemicals, Raw Manufacturing   56/94        44.0
 SAU  2021      RS   False                       Energy Drilling and Mining, Chemicals, Raw Manufacturing   56/94        44.0
 SAU  2022      R0     NaN                                                            (not in lab sample)     NaN     

**Takeaway (C): the check passes for Saudi Arabia.** Under RS, its 2019, 2020 and 2023 baskets are all {Energy, Chemicals, Raw Manufacturing}. That is the same as its 2021 basket, which needed no fix. Its rank goes from 4/105, 4/105 and 3/109 to **48/105, 49/105 and 55/109** (ATLAS 45, 46, 49; 2021: 56/94, ATLAS 44). **Qatar is in the lab sample only in 2022.** There its basket goes from {Energy, aggregate, Services} to {Energy} alone, and its rank from 11 to **104/105** (ATLAS 57). There is no clean Qatar year in the lab sample to check this against.

### D. Rule A on the joint common sample

In [5]:
# D. Rule A on the joint common sample (pre-stated, unchanged since 11-19)
rows = []
for y in YEARS:
    for v in VN:
        c = CS[y, v]; e, e0 = VAR[y, v][c], VAR[y, V0][c]
        m1, m2 = stats.spearmanr(e, ATL[y][c])[0], stats.spearmanr(e, LNY[y][c])[0]
        rows.append({'year': y, 'variant': v, 'n': len(c), 'M1 rho ATLAS': m1, 'M2 rho lnGDPpc': m2,
                     'M1 diff': m1 - stats.spearmanr(e0, ATL[y][c])[0], 'M2 diff': m2 - stats.spearmanr(e0, LNY[y][c])[0],
                     'rho w/ R0': stats.spearmanr(e, e0)[0],
                     'SAU': f"{int(rank(e, c)['SAU'])}" if 'SAU' in c else '-', 'QAT': f"{int(rank(e, c)['QAT'])}" if 'QAT' in c else '-'})
P = pd.DataFrame(rows); print(P.round(3).to_string(index=False))
P.round(4).to_csv(FIG / 'primary_metrics.csv', index=False)
print('ATLAS ranks in the common sample: SAU', {y: int(rank(ATL[y], CS[y, V0])['SAU']) if 'SAU' in CS[y, V0] else '-' for y in YEARS},
      '| QAT', {y: int(rank(ATL[y], CS[y, V0])['QAT']) if 'QAT' in CS[y, V0] else '-' for y in YEARS})

 year                            variant   n  M1 rho ATLAS  M2 rho lnGDPpc  M1 diff  M2 diff  rho w/ R0 SAU QAT
 2019           R0 lab mapping (rebuilt) 105         0.689           0.453    0.000    0.000      1.000   4   -
 2019  F1 ch.89 -> Transportation (ref.) 105         0.726           0.487    0.037    0.034      0.983   2   -
 2019 RS reassign, switch rule (primary) 105         0.714           0.445    0.025   -0.008      0.977  48   -
 2019       RM reassign, mechanical rule 105         0.714           0.445    0.025   -0.008      0.977  48   -
 2019     RK reassign, Saudi Arabia only 105         0.714           0.445    0.025   -0.008      0.977  48   -
 2019                              F1+RS 105         0.743           0.462    0.054    0.009      0.953  48   -
 2020           R0 lab mapping (rebuilt) 105         0.760           0.508    0.000    0.000      1.000   4   -
 2020  F1 ch.89 -> Transportation (ref.) 105         0.787           0.519    0.026    0.011      0.973 

**Takeaway (D):** RS raises M1 in 2019 (0.689 → 0.714) and in 2022 (+0.013). It lowers M2 in 2019 (0.453 → 0.445), 2020 (−0.046), 2022 (−0.046) and 2023 (−0.028). RS, RM and RK are identical in 2019–2021 because they flag the same in-sample reporter (Saudi Arabia), or nobody. They differ only in 2022: RS flags Qatar, RM flags Albania and Bermuda (both outside the common sample), and RK flags nobody. F1+RS has the highest M1 of all variants in every year (tied with F1 in 2021, when RS flags nobody; 2019: 0.743). Its M2 is below F1's in every year except 2021, where they are equal.

In [6]:
# D2. Rule A verdicts + 2019 paired country bootstrap (2,000 draws)
rng = np.random.default_rng(20261126); y = 2019; B = 2000
dec = []
for v in VN[1:]:
    c = np.array(CS[y, v]); at, ly = ATL[y][c].to_numpy(), LNY[y][c].to_numpy()
    e0, e1 = VAR[y, V0][c].to_numpy(), VAR[y, v][c].to_numpy()
    bb = np.array([(stats.spearmanr(e1[i], at[i])[0] - stats.spearmanr(e0[i], at[i])[0],
                    stats.spearmanr(e1[i], ly[i])[0] - stats.spearmanr(e0[i], ly[i])[0]) for i in rng.integers(0, len(c), size=(B, len(c)))])
    s = P[P.variant == v].set_index('year'); q1, q2 = np.percentile(bb[:, 0], [2.5, 97.5]), np.percentile(bb[:, 1], [2.5, 97.5])
    both = (s['M1 diff'] > 0) & (s['M2 diff'] > 0)
    dec.append({'variant': v, 'M1 diff 2019': round(s.loc[2019, 'M1 diff'], 3), 'M1 95% CI': f'[{q1[0]:.3f}, {q1[1]:.3f}]',
                'M2 diff 2019': round(s.loc[2019, 'M2 diff'], 3), 'M2 95% CI': f'[{q2[0]:.3f}, {q2[1]:.3f}]',
                'both 2019': bool(both[2019]), 'both, of 2020-22': int(both.loc[[2020, 2021, 2022]].sum()),
                'BETTER (rule A)': bool(both[2019] and both.loc[[2020, 2021, 2022]].sum() >= 2)})
DEC = pd.DataFrame(dec); print(DEC.to_string(index=False))
print('\nnote: years in which a rule flags nobody in the sample give diff = 0 exactly (counted as "not both > 0").')
print('flagged reporters inside the common sample:', {r: {yy: [eci.m49_to_iso3(m) for m in FLAG[r][yy] if eci.m49_to_iso3(m) in CS[yy, V0]] for yy in YEARS} for r in FLAG})

                           variant  M1 diff 2019       M1 95% CI  M2 diff 2019       M2 95% CI  both 2019  both, of 2020-22  BETTER (rule A)
 F1 ch.89 -> Transportation (ref.)         0.037 [-0.001, 0.096]         0.034  [0.005, 0.080]       True                 3             True
RS reassign, switch rule (primary)         0.025  [0.002, 0.051]        -0.008 [-0.051, 0.032]      False                 0            False
      RM reassign, mechanical rule         0.025  [0.002, 0.053]        -0.008 [-0.049, 0.031]      False                 1            False
    RK reassign, Saudi Arabia only         0.025  [0.001, 0.051]        -0.008 [-0.050, 0.029]      False                 0            False
                             F1+RS         0.054  [0.001, 0.116]         0.009 [-0.049, 0.072]       True                 1            False

note: years in which a rule flags nobody in the sample give diff = 0 exactly (counted as "not both > 0").
flagged reporters inside the common sample: {'R

**Takeaway (D2): every reassignment variant fails rule A.** RS, RK: 0 of 3 check years with both metrics up. RM: 1 of 3 (2022, where it only touches countries outside the sample). F1+RS: 1 of 3 (2021, which is F1's own effect because RS flags nobody). The RS 2019 M1 gain has a CI that excludes 0 ([0.002, 0.051]), but the M2 change does not ([−0.051, 0.032]). **F1 passes again** (+0.037 [−0.001, 0.096], +0.034 [0.005, 0.080]; new seed, so the CIs differ slightly from 11-25).

In [7]:
# D3. Pre-stated robustness: is any metric change more than the flagged reporters' own rank change?
# Score each variant with the flagged reporters themselves EXCLUDED from the Spearman (their ranks still shape everyone's ECI).
rows = []
for y in CHK:
    for v in VN[2:]:
        rule = VARS[v][1]; fl = [eci.m49_to_iso3(m) for m in FLAG[rule][y]]
        c = [i for i in CS[y, v] if i not in fl]; e, e0 = VAR[y, v][c], VAR[y, V0][c]
        rows.append({'year': y, 'variant': v.split()[0], 'excluded': ', '.join(fl) or '-', 'n': len(c),
                     'M1 diff': stats.spearmanr(e, ATL[y][c])[0] - stats.spearmanr(e0, ATL[y][c])[0],
                     'M2 diff': stats.spearmanr(e, LNY[y][c])[0] - stats.spearmanr(e0, LNY[y][c])[0],
                     'others: max |rank change|': int((rank(e, c) - rank(e0, c)).abs().max()),
                     'others: n moving >= 5': int(((rank(e, c) - rank(e0, c)).abs() >= 5).sum())})
print(pd.DataFrame(rows).round(3).to_string(index=False))

 year variant excluded   n  M1 diff  M2 diff  others: max |rank change|  others: n moving >= 5
 2019      RS      SAU 104    0.020    0.005                         23                     23
 2019      RM ALB, SAU 104    0.020    0.005                         23                     23
 2019      RK      SAU 104    0.020    0.005                         23                     23
 2019   F1+RS      SAU 104    0.049    0.022                         54                     37
 2020      RS QAT, SAU 104   -0.007   -0.036                         22                     15
 2020      RM      SAU 104   -0.007   -0.036                         22                     15
 2020      RK      SAU 104   -0.007   -0.036                         22                     15
 2020   F1+RS QAT, SAU 104    0.026   -0.009                         55                     35
 2021      RS        -  94    0.000    0.000                          0                      0
 2021      RM      ALB  94    0.000    0.000      

**Takeaway (D3):** with the flagged reporters left out of the Spearman, RS's 2019 effect on everyone else is positive on both metrics (+0.020 / +0.005), and so is 2022 (+0.012 / +0.004). 2020 stays negative (−0.007 / −0.036). So in 2019 and 2022 the M2 loss comes from Saudi Arabia and Qatar themselves moving down. In 2020 it comes from the side effects on other countries (see E). The side effects are not small: in 2019, 23 other countries move ≥5 ranks (largest move 23). In 2020 the count is 15 (largest 22).

### E. Side effects through the world denominator

In [8]:
# E. Side effects through the world denominator: M cells that flip vs R0, inside vs outside the flagged reporters
rows = []
for y in YEARS:
    v = 'RS reassign, switch rule (primary)'; fl = [eci.m49_to_iso3(m) for m in FLAG['RULE_SWITCH'][y]]
    A, Bm = MM[y, V0], MM[y, v]; idx = sorted(set(A.index) & set(Bm.index)); cols = sorted(set(A.columns) & set(Bm.columns))
    D = (A.loc[idx, cols] != Bm.loc[idx, cols])
    X0 = XV[y, V0].loc[KEEP[y, v]]; X1 = XV[y, v].loc[KEEP[y, v]]
    rows.append({'year': y, 'flagged': ', '.join(fl) or '-', 'world Services $B R0 -> RS': f'{X0[SVC].sum()/1e9:,.0f} -> {X1[SVC].sum()/1e9:,.0f}',
                 'world Energy $B R0 -> RS': f'{X0[ENE].sum()/1e9:,.0f} -> {X1[ENE].sum()/1e9:,.0f}',
                 'world 18-col total change': f'{X1.to_numpy().sum()/X0.to_numpy().sum()-1:+.2%}',
                 'M flips, flagged rows': int(D.loc[[i for i in idx if i in fl]].to_numpy().sum()),
                 'M flips, other rows': int(D.loc[[i for i in idx if i not in fl]].to_numpy().sum()),
                 'other countries with a flip': ', '.join(i for i in idx if i not in fl and D.loc[i].any())})
FL = pd.DataFrame(rows); print(FL.to_string(index=False))
# 2019 detail: which non-flagged cells flip, with RCA before -> after
y = 2019; v = 'RS reassign, switch rule (primary)'
r0, r1 = eci.rca(XV[y, V0].loc[KEEP[y, v]]), eci.rca(XV[y, v].loc[KEEP[y, v]])
fl = [eci.m49_to_iso3(m) for m in FLAG['RULE_SWITCH'][y]]
det = [(i, g, r0.at[i, g], r1.at[i, g]) for i in r0.index if i not in fl for g in r0.columns if (r0.at[i, g] > 1) != (r1.at[i, g] > 1)]
print('\n2019 non-flagged flips:'); print(pd.DataFrame(det, columns=['iso3', 'group', 'RCA R0', 'RCA RS']).round(3).to_string(index=False))
print('\n2019 group PCI rank (1 = most complex), R0 vs RS:')
pr = pd.DataFrame({'R0': PCI[y, V0].rank(ascending=False), 'RS': PCI[y, v].rank(ascending=False)}).astype('Int64').sort_values('R0'); print(pr.to_string())

 year  flagged world Services $B R0 -> RS world Energy $B R0 -> RS world 18-col total change  M flips, flagged rows  M flips, other rows                                               other countries with a flip
 2019      SAU                 821 -> 619           2,118 -> 2,321                    -0.95%                      4                   16 BGR, DEU, ESP, EST, FIN, GBR, KOR, LBN, NZL, ROU, SGP, SVN, SWE, USA, ZWE
 2020 QAT, SAU                 668 -> 535           1,562 -> 1,695                    -0.66%                      3                   11                     AZE, BGR, CYP, ESP, EST, ISR, MLT, PAN, PRT, ROU, ZMB
 2021        -                 735 -> 735           2,656 -> 2,656                    +0.00%                      0                    0                                                                          
 2022      QAT                 898 -> 797           3,419 -> 3,519                    -0.37%                      2                   10                    


2019 non-flagged flips:
iso3                         group  RCA R0  RCA RS
 BGR        Services and Utilities   0.790   1.038
 DEU        Services and Utilities   0.770   1.012
 ESP                 Wood Products   1.006   0.996
 EST    Energy Drilling and Mining   1.098   0.993
 EST        Services and Utilities   0.770   1.012
 FIN      High-complexity Products   0.990   1.016
 GBR        Services and Utilities   0.815   1.071
 KOR      High-complexity Products   0.984   1.010
 LBN             Raw Manufacturing   1.009   0.999
 NZL        Services and Utilities   0.764   1.005
 ROU      High-complexity Products   0.989   1.016
 SGP    Energy Drilling and Mining   1.094   0.989
 SVN                Transportation   1.002   0.993
 SWE        Services and Utilities   0.783   1.030
 USA    Energy Drilling and Mining   1.021   0.923
 ZWE Vegetables, Fruits and Plants   1.002   0.993

2019 group PCI rank (1 = most complex), R0 vs RS:
                               R0  RS
Transportation     

**Takeaway (E):** removing Saudi Arabia's $202B from 999999 shrinks the **world *Services* column by a quarter** in 2019 ($821B → $619B). Six countries then cross RCA = 1 in *Services*: Germany 0.770 → 1.012, UK 0.815 → 1.071, Sweden, New Zealand, Bulgaria and Estonia. Adding the same $202B to *Energy* pushes Estonia, Singapore and the USA out of their *Energy* cells (the USA goes 1.021 → 0.923). Overall, 16 M cells flip outside Saudi Arabia's own row in 2019 (11 in 2020, 10 in 2022, 9 in 2023), against 4 inside it. The world total over the 18 columns changes by only −0.95% (11-25's F4: −3.8%), but the effect is concentrated in two columns. Group PCI ranks barely move: Services 8 → 7, Energy 17 → 18.

In [9]:
# E2. Largest rank movers under RS, by year (negative = up)
for y in CHK:
    v = 'RS reassign, switch rule (primary)'; c = CS[y, v]; d = (rank(VAR[y, v], c) - rank(VAR[y, V0], c)).sort_values()
    big = d[d.abs() >= 3]
    print(y, 'n moving >= 3 ranks:', len(big), '|', ', '.join(f'{i} {int(rank(VAR[y, V0], c)[i])}->{int(rank(VAR[y, v], c)[i])}' for i in big.index) or '-')

2019 n moving >= 3 ranks: 42 | NZL 87->64, FIN 23->10, EST 45->33, SGP 25->14, ROU 22->13, USA 30->22, LKA 68->60, ARG 75->68, KHM 47->41, LUX 59->54, NIC 96->91, PAK 60->55, BGR 49->44, LVA 65->61, IDN 66->62, ETH 85->81, UKR 77->73, TZA 94->90, KOR 11->7, ISL 78->75, GRC 73->70, SWZ 46->43, UZB 83->86, MOZ 81->84, SEN 93->96, BIH 16->19, NGA 18->21, ZMB 54->58, ZAF 84->88, BOL 90->94, COG 41->46, JAM 44->49, COD 34->39, MLT 40->45, PAN 19->24, TTO 51->57, KAZ 69->76, LBN 57->65, OMN 58->67, KWT 63->79, BRN 63->79, SAU 4->48
2020 n moving >= 3 ranks: 42 | ZMB 75->53, ROU 24->13, ISR 28->19, CYP 53->46, BGR 39->33, VNM 33->27, IDN 66->61, EST 45->40, TUN 26->21, NZL 76->72, UZB 69->65, MYS 27->23, MDG 91->87, KEN 67->63, KGZ 88->84, EGY 68->64, POL 38->35, CHN 14->11, GTM 47->44, ISL 74->71, SRB 40->37, NGA 50->47, NLD 21->24, NOR 64->67, SWE 23->26, IRL 12->15, JOR 63->66, ARG 65->68, AZE 99->102, PRT 42->45, PAN 19->22, KAZ 70->73, AUS 92->96, BEL 35->39, COD 29->34, SEN 84->89, JAM 

2022 n moving >= 3 ranks: 32 | CAN 73->56, EST 45->34, SWE 25->17, MLT 23->15, LUX 50->44, NZL 76->70, LVA 51->46, FRA 35->30, KEN 47->43, GTM 49->45, BFA 84->80, SUR 83->79, SGP 8->4, ETH 78->74, MDG 67->64, AZE 105->102, MYS 31->28, NPL 66->63, ROU 28->25, HKG 32->29, TZA 85->82, UKR 53->50, SVN 14->11, EGY 65->68, JOR 63->66, ISR 18->21, LTU 46->49, PRY 72->75, BEL 43->47, ARG 44->51, BRN 75->84, QAT 11->104


**Takeaway (E2):** 42 countries move ≥3 ranks in 2019 (42 in 2020, 32 in 2022, 0 in 2021). Besides the flagged reporters, the large moves are New Zealand 87 → 64, Finland 23 → 10, Singapore 25 → 14 and Romania 22 → 13 (2019); Zambia 75 → 53 (2020); Canada 73 → 56 (2022). Kuwait and Brunei fall 63 → 79 in 2019 and 59 → 75 in 2020, presumably because the world *Energy* column now carries Saudi Arabia's oil, which lowers their *Energy* RCA relative to the other cells (not checked cell by cell tonight).

### F. Spring 2026 report countries

In [10]:
# F. Spring 2026 report countries: ranks, and position among (country + 3 peers), 2019-2022
rows = []
for y in CHK:
    for iso, nm in R.REPORT_COUNTRIES.items():
        r = {'year': y, 'country': iso}
        for v in [V0, 'RS reassign, switch rule (primary)', 'F1+RS']:
            cv = CS[y, v]; grp = [i for i in [iso] + PEERS[iso] if i in cv]
            r[v.split()[0] + ' rank'] = int(rank(VAR[y, v], cv)[iso]) if iso in cv else None
            r[v.split()[0] + ' pos'] = int(VAR[y, v][grp].rank(ascending=False)[iso]) if iso in grp else None
            r['peers in sample'] = len(grp) - 1
        rows.append(r)
RP = pd.DataFrame(rows); print(RP.to_string(index=False))
ch = RP[(RP['R0 pos'] != RP['RS pos']) | (RP['R0 pos'] != RP['F1+RS pos'])]
print('\nchanged peer positions (R0 vs RS or F1+RS):', len(ch), 'of', len(RP)); print(ch.to_string(index=False) if len(ch) else '-')
# UAE peer group in detail: ordering of ARE, SAU, KWT, QAT
print('\nUAE peer-group ordering (1 = highest grouped ECI):')
for y in CHK:
    for v in [V0, 'RS reassign, switch rule (primary)']:
        cv = CS[y, v]; grp = [i for i in ['ARE', 'SAU', 'KWT', 'QAT'] if i in cv]
        o = VAR[y, v][grp].sort_values(ascending=False)
        at = ATL[y][grp].sort_values(ascending=False)
        print(y, v.split()[0].ljust(3), 'grouped:', ' > '.join(o.index), '| ATLAS:', ' > '.join(at.index))

 year country  R0 rank  R0 pos  peers in sample  RS rank  RS pos  F1+RS rank  F1+RS pos
 2019     CAN       82       3                3       83       3          80          3
 2019     DEU        1       1                3        2       1           2          1
 2019     POL       33       4                3       32       4          21          4
 2019     SGP       25       4                3       14       4          15          4
 2019     ARE      104       3                2      105       3         105          3
 2019     GBR       35       4                3       34       4          33          4
 2019     VNM       31       3                2       29       3          30          3
 2020     CAN       72       3                3       70       3          69          3
 2020     DEU        2       1                3        2       1           1          1
 2020     POL       38       4                3       35       4          33          4
 2020     SGP       16       4  

**Takeaway (F):** under RS, **1 of 28** report-country peer positions changes in 2019–2022: UAE 2nd → 1st in 2022, because Qatar drops to 104. Under F1+RS, 4 change (that one, plus Singapore 4th → 3rd in 2020 and 2021 and 4th → 1st in 2022). The UAE peer ordering in 2019–2021 is the same under R0 and RS (SAU > KWT > ARE) and the opposite of ATLAS (ARE > SAU > KWT). So even after the fix, grouped ECI still puts Saudi Arabia and Kuwait above the UAE. Singapore moves up (2019: 25 → 14; 2022: 8 → 4), the UK falls in 2020 (22 → 29), and Canada rises in 2022 (73 → 56).

### G. (Post hoc) Why does fixing the oil reporters hurt income fit?

In [11]:
# G (added after seeing D/D3, labelled post hoc). Why does income fit (M2) fall when Saudi Arabia / Qatar are fixed?
# Compare each flagged reporter's grouped rank with its income rank, ATLAS rank and our own HS4 rank (HS4 base excludes ch.98/99).
rows = []
for y in CHK:
    c = CS[y, V0]
    for iso in ['SAU', 'QAT']:
        if iso not in c: continue
        rows.append({'year': y, 'iso3': iso, 'n': len(c), 'lnGDPpc rank': int(LNY[y][c].rank(ascending=False)[iso]),
                     'R0 rank': int(rank(VAR[y, V0], c)[iso]), 'RS rank': int(rank(VAR[y, 'RS reassign, switch rule (primary)'], c)[iso]),
                     'ATLAS rank': int(rank(ATL[y], c)[iso]), 'HS4 rank': int(rank(HB[y][0], c)[iso]),
                     'flagged': iso in [eci.m49_to_iso3(m) for m in FLAG['RULE_SWITCH'][y]]})
print(pd.DataFrame(rows).to_string(index=False))
# Spearman of income rank with ATLAS on the same sample, for reference: how much of the M2 ceiling is the resource-rich pattern?
print('Spearman(ATLAS ECI, lnGDPpc) on the common sample:', {y: round(stats.spearmanr(ATL[y][CS[y, V0]], LNY[y][CS[y, V0]])[0], 3) for y in CHK})

 year iso3   n  lnGDPpc rank  R0 rank  RS rank  ATLAS rank  HS4 rank  flagged
 2019  SAU 105            15        4       48          45        31     True
 2020  SAU 105            16        4       49          46        35     True
 2021  SAU  94            14       56       56          44        28    False
 2022  QAT 105             4       11      104          57        34     True
Spearman(ATLAS ECI, lnGDPpc) on the common sample: {2019: np.float64(0.819), 2020: np.float64(0.806), 2021: np.float64(0.768), 2022: np.float64(0.749)}


**Takeaway (G, added after seeing D/D3):** the flagged reporters are much richer than any complexity measure says. Saudi Arabia ranks 15–16 of 105 on log GDPpc in 2019–2020, but 45–46 on ATLAS and 31–35 on our HS4 ECI. Qatar ranks 4 of 105 on income in 2022, but 57 on ATLAS and 34 on HS4. R0's data artefact puts them near their *income* rank (4, 4, 11), so M2 rewards the artefact. In this series, M2 cannot tell "ECI agrees with income because the data are wrong" from "ECI agrees with income because it measures capability" for resource-rich countries. On the same sample, ATLAS itself correlates 0.819 / 0.806 / 0.768 / 0.749 with log GDPpc (2019–2022).

### Figure

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)
COL = dict(zip(VN[1:], ['#9a9994', '#2a78d6', '#eb6834', '#7b5cc4', '#1baf7a']))
for ax, m, t in zip(axes, ['M1 diff', 'M2 diff'], ['ΔM1 vs R0 (Spearman with ATLAS ECI)', 'ΔM2 vs R0 (Spearman with log GDPpc)']):
    for v in VN[1:]:
        s = P[P.variant == v]; ax.plot(s.year, s[m], color=COL[v], lw=2, marker='o', ms=5, label=v, ls='--' if v == VN[1] else '-')
    ax.axhline(0, color='#bdbcb8', lw=0.8); ax.axvspan(2022.5, 2023.5, color='#efefec', zorder=0)
    ax.set_title(t, fontsize=10, loc='left'); ax.set_xticks(YEARS); ax.grid(axis='y', color='#ececea', lw=0.8)
    for s_ in ['top', 'right']: ax.spines[s_].set_visible(False)
axes[0].set_ylabel('Δ Spearman ρ'); axes[1].legend(frameon=False, fontsize=7.5)
fig.text(0.99, 0.01, 'shaded: 2023 (suspect, D1)', ha='right', fontsize=7.5, color='#52514e')
fig.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(FIG / 'delta_metrics.png', dpi=100); plt.close(fig)
print('saved figures/delta_metrics.png')

saved figures/delta_metrics.png


![delta metrics](figures/delta_metrics.png)

**Takeaway (figure):** F1 (grey, dashed) is the only variant at or above zero on both panels in 2019–2022. The reassignment variants (blue, orange, purple; identical in 2019–2021) gain on M1 in 2019 but dip on M2 in 2020 and 2022. F1+RS (green) has the highest M1 but falls below zero on M2 in 2020 and 2022.

## Caveats

- **Rule A is the lab's pre-stated yardstick, and the verdict is "fails".** G (post hoc) argues that M2 penalises a correct data fix for resource-rich reporters. That is an argument for *changing the yardstick for data corrections*, not a reason to re-read tonight's verdict as a pass.
- **Without mirror data, we don't know that all of 999999 is oil.** For Saudi Arabia, the 2021 basket check is strong indirect evidence. For Qatar there is no clean in-sample year, and its 999999 may include LNG plus other goods. Moving *all* of it to *Energy* may over-correct (rank 104 vs ATLAS 57).
- **The sample is fixed by R0's filter.** A cleaning-step version would interact with drop-any-zero (B), and the result would differ.
- **Small samples, short window**, and ATLAS is a benchmark, not ground truth. Nominal values. 2023 is suspect (D1). Saudi Arabia is absent from the 2022 raw pull, so the 2022 check year tests Qatar only.
- The peer dict is the one used by earlier runs (from the Spring 2026 PDFs). It was not re-read tonight.

## Helpful to know

- **A blind "ch. 27 = 0" rule is the wrong detector.** It misses Qatar, which files some ch. 27 in its 999999 years, and it catches Albania and Bermuda, which never file oil. The "convention switch" rule (999999 ≥ 50% *and* ch. 27 ≥ 50% in another year) reproduces 11-25's hand list exactly.
- **Reassigning 999999 empties the *Services* column for that reporter**, and the lab's drop-any-zero filter then removes the country. Whoever implements this in cleaning must handle the filter too.
- **The *Services* column is a small column dominated by a few reporters.** Saudi Arabia alone is a quarter of world 999999 in 2019. Moving it flips *Services* RCA for six rich European/Oceanian countries.
- **M2 (income fit) rewards errors that push rich oil exporters up.** Any future data-correction variant that touches Gulf reporters will look worse on M2 even when it is right. Score data corrections with M1 and the built-in own-year check, and report M2 separately.
- **F1 survives a third seed.** It is still the only variant to pass rule A.

## New research avenues (also in `proposals/2026-11-26.md`)

1. **A rule for data corrections that is separate from rule A.** Pre-state a "rule D" for fixes to the *data* (not the method): pass if (i) an own-year consistency check passes (like Saudi Arabia 2021) and (ii) M1 doesn't fall, with M2 reported but not decisive, because of the resource-rich pattern in G. Re-score RS and 11-25's F4 under it. One night, no new computation beyond tonight's.
2. **Partial reassignment for Qatar.** Move only the part of 999999 that brings Qatar's 2020/2022 ch. 27 *share* up to its own 2019/2021/2023 average (about 0.86). Check whether its 2022 rank lands closer to ATLAS 57 than to 104. One night.
3. **Resource-adjusted income residuals with RS** (11-21 #1). Fit lnGDPpc ~ HS4 ECI + resource share. Then check whether RS's M2 loss disappears when M2 is measured on the residual, i.e. with resource-rich income netted out. This directly tests G's explanation. One night.
4. **Why grouped ECI still puts Saudi Arabia and Kuwait above the UAE** (F). Inspect the UAE's 2019–2021 18-group RCA row, including its re-export cells, against Kuwait's {Energy, …} basket. The UAE report's peer sentence depends on it. One night.

## Data issues / data needed

- No new data request. Tonight's result strengthens `data-requests/2026-11-25-999999-reporters.md` (in the open 11-25 PR): mirror imports of HS 2709/2711 from Saudi Arabia (682) and Qatar (634) would show how much of each 999999 line is oil, and that is what decides between a full and a partial reassignment (avenue 2).
- Saudi Arabia is absent from the 2022 raw pull (known).

In [13]:
print(f'Run time: {time.time() - T0:.1f} s')

Run time: 13.7 s
